# Week 88, Day 1: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')

**Predict before you run:** what will this print?



In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE saved_route (name TEXT UNIQUE COLLATE NOCASE)")
conn.execute("INSERT INTO saved_route VALUES ('Morning loop')")
for name in ["Ridge walk", "morning LOOP"]:
    try:
        conn.execute("INSERT INTO saved_route VALUES (?)", (name,))
        print("saved", name)
    except sqlite3.IntegrityError as e:
        print("refused", name, "-", e)
print(conn.execute("SELECT name FROM saved_route WHERE name = 'MORNING loop'").fetchall())

In [ ]:
import sqlite3

MIGRATIONS = [
    "CREATE TABLE junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL)",
    "CREATE TABLE saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE)",
]

def migrate(conn):
    conn.execute("CREATE TABLE IF NOT EXISTS schema_version (version INTEGER NOT NULL)")
    done = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_version").fetchone()[0]
    for number, sql in enumerate(MIGRATIONS, start=1):
        if number > done:
            with conn:
                conn.execute(sql)
                conn.execute("INSERT INTO schema_version VALUES (?)", (number,))
            print("applied", number)

conn = sqlite3.connect(":memory:")
migrate(conn)
migrate(conn)
print([r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")])

> **Key idea.** Change live schemas with numbered migrations recorded in a `schema_version` table and applied in order on start-up; never edit an applied migration; run each in a transaction (statement by statement, not `executescript`) so it's all or nothing. Model ordered children with a position column in the primary key. Use `COLLATE NOCASE` for names compared without capitals. Repositories speak the domain's language and translate database errors into domain exceptions. Validate form at the edge and meaning in the domain and service; return plain data to interfaces, which map exceptions to status codes (400, 404, 409).

### Exercise: Valid route names

Write `clean_name(name)`: strip the ends and collapse runs of spaces to one; then raise `ValueError` with `"name is required"` if it's empty, `"name is too long"` if it has more than 40 characters, or `"name has invalid characters"` unless it contains only letters, digits, spaces, hyphens and apostrophes (letters may be accented: use `str.isalpha`). Return the cleaned name.

In [ ]:
def clean_name(name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"tidy, then check\"\n    assert clean_name(\"  Ridge   walk \") == \"Ridge walk\" and clean_name(\"Kate's caf\u00e9 loop-2\") == \"Kate's caf\u00e9 loop-2\"\n    for bad, message in ((\"   \", \"name is required\"), (\"x\" * 41, \"name is too long\"), (\"Ridge <walk>\", \"name has invalid characters\"),\n                         (\"a/b\", \"name has invalid characters\")):\n        try:\n            clean_name(bad)\n        except ValueError as e:\n            assert str(e) == message, str(e)\n        else:\n            raise AssertionError(bad)\n    assert clean_name(\"y\" * 40) == \"y\" * 40")

### Exercise: Migrations

Write `current_version(conn)`: the highest version in `schema_version` (creating the table, a single `version INTEGER` column, if it doesn't exist), or 0. Then `migrate(conn, migrations)`: `migrations` is a list of SQL strings, numbered from 1; apply each one above the current version, in order, each together with its version row in one transaction (`with conn:`), and return the list of numbers applied.

In [ ]:
import sqlite3

def current_version(conn):
    ___

def migrate(conn, migrations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\nM = [\"CREATE TABLE junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL)\",\n     \"CREATE TABLE saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE)\",\n     \"CREATE TABLE saved_stop (route_id INTEGER NOT NULL REFERENCES saved_route(id), position INTEGER NOT NULL, junction TEXT NOT NULL, PRIMARY KEY (route_id, position))\"]\n\ndef _tables(conn):\n    return [r[0] for r in conn.execute(\"SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name\")]\n\ndef test_migrate():\n    \"in order, once each\"\n    conn = sqlite3.connect(\":memory:\")\n    assert current_version(conn) == 0\n    assert migrate(conn, M[:1]) == [1] and current_version(conn) == 1\n    assert migrate(conn, M) == [2, 3] and migrate(conn, M) == []\n    assert _tables(conn) == [\"junction\", \"saved_route\", \"saved_stop\", \"schema_version\"]\n    assert current_version(conn) == 3")

### Exercise: A saved-route repository

Write `RouteRepository(conn)`, creating its two tables if needed (`saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE)` and `saved_stop (route_id INTEGER NOT NULL, position INTEGER NOT NULL, junction TEXT NOT NULL, PRIMARY KEY (route_id, position))`). `add(name, stops)` stores both in one transaction and returns the id, raising the domain's `DuplicateName(name)` (given) if the name is taken, ignoring capitals; `get(name)` returns `(stored_name, stops)` or `None`, ignoring capitals; `names()` lists names in alphabetical order ignoring capitals; `delete(name)` removes the route and its stops and returns whether there was one.

In [ ]:
import sqlite3

class DuplicateName(Exception):
    pass

class RouteRepository:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\ndef test_repository():\n    \"names, order, capitals\"\n    repo = RouteRepository(sqlite3.connect(\":memory:\"))\n    first = repo.add(\"Ridge walk\", [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    repo.add(\"canal stroll\", [\"Mill\", \"Lake\"])\n    assert repo.get(\"RIDGE WALK\") == (\"Ridge walk\", [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    assert repo.names() == [\"canal stroll\", \"Ridge walk\"] and repo.get(\"nothing\") is None\n    try:\n        repo.add(\"ridge walk\", [\"Mill\"])\n    except DuplicateName as e:\n        assert e.args[0] == \"ridge walk\"\n    else:\n        raise AssertionError(\"duplicate\")\n    assert repo.conn.execute(\"SELECT COUNT(*) FROM saved_stop\").fetchone() == (6,), \"a refused route stores no stops\"\n    assert repo.delete(\"Ridge Walk\") is True and repo.delete(\"Ridge Walk\") is False\n    assert repo.conn.execute(\"SELECT COUNT(*) FROM saved_stop\").fetchone() == (2,) and isinstance(first, int)")

### Exercise: Saving only walkable routes

Using `waymark_domain` and `waymark_map` (the page files) and the given `RouteRepository` and `clean_name`, write `save_route(repo, network, name, stops)`: clean the name; require at least two stops (`ValueError("a route needs at least two stops")`); raise the domain's `UnknownPlace(stop)` for the first unknown stop and `NoRoute("no path from A to B")` for the first consecutive pair with no direct path; then save and return `{"name", "stops", "km", "climb_m", "minutes"}` with `km` rounded to 1 place.

In [ ]:
import sqlite3
from waymark_domain import NoRoute, Route, UnknownPlace
from waymark_map import sample_network

class DuplicateName(Exception):
    pass

def clean_name(name):
    cleaned = " ".join(name.split())
    if not cleaned or len(cleaned) > 40 or not all(c.isalpha() or c.isdigit() or c in " -'" for c in cleaned):
        raise ValueError("bad name")
    return cleaned

class RouteRepository:
    def __init__(self, conn):
        self.conn = conn
        conn.execute("CREATE TABLE IF NOT EXISTS saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE)")
        conn.execute("CREATE TABLE IF NOT EXISTS saved_stop (route_id INTEGER, position INTEGER, junction TEXT, PRIMARY KEY (route_id, position))")
    def add(self, name, stops):
        try:
            with self.conn:
                cur = self.conn.execute("INSERT INTO saved_route (name) VALUES (?)", (name,))
                self.conn.executemany("INSERT INTO saved_stop VALUES (?, ?, ?)", [(cur.lastrowid, i, s) for i, s in enumerate(stops)])
        except sqlite3.IntegrityError:
            raise DuplicateName(name)
        return cur.lastrowid

def save_route(repo, network, name, stops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\ndef test_save():\n    \"S3's three scenarios\"\n    repo, network = RouteRepository(sqlite3.connect(\":memory:\")), sample_network()\n    assert save_route(repo, network, \" Ridge  walk\", [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]) == {\n        \"name\": \"Ridge walk\", \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"km\": 6.0, \"climb_m\": 270, \"minutes\": 99}\n    for stops, kind, message in (([\"Mill\", \"Ridge\"], NoRoute, \"no path from Mill to Ridge\"), ([\"Mill\", \"Moon\"], UnknownPlace, \"Moon\"),\n                                 ([\"Mill\"], ValueError, \"a route needs at least two stops\")):\n        try:\n            save_route(repo, network, \"Another\", stops)\n        except kind as e:\n            assert e.args[0] == message\n        else:\n            raise AssertionError(stops)\n    try:\n        save_route(repo, network, \"ridge WALK\", [\"Mill\", \"Weir\"])\n    except DuplicateName:\n        pass\n    else:\n        raise AssertionError(\"duplicate\")")

### Exercise: The saved-routes API

Write `handle(method, path, body, repo, network)` using the given `save_route`: `POST /saved-routes` with JSON `{"name", "stops"}` gives `201` and the result; `GET /saved-routes` gives `200` and `{"names": [...]}`; `GET /saved-routes/NAME` (URL-decoded) gives `200` and `{"name", "stops"}` or `404`; `DELETE /saved-routes/NAME` gives `204` with `None` or `404`. Errors: invalid JSON, a missing or wrongly typed field, or a `ValueError` give `400`; `UnknownPlace` and `NoRoute` give `404`; `DuplicateName` gives `409`; each with `{"error": message}` (for the 404s from the domain, the exception's message; for 409, `"name taken"`; for a missing route, `"not found"`). Any other path is `404` with `"not found"`.

In [ ]:
import json
import sqlite3
from urllib.parse import unquote
from waymark_domain import NoRoute, UnknownPlace
from waymark_map import sample_network

class DuplicateName(Exception):
    pass

class RouteRepository:
    def __init__(self, conn):
        self.conn = conn
        conn.execute("CREATE TABLE IF NOT EXISTS saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE)")
        conn.execute("CREATE TABLE IF NOT EXISTS saved_stop (route_id INTEGER, position INTEGER, junction TEXT, PRIMARY KEY (route_id, position))")
    def add(self, name, stops):
        try:
            with self.conn:
                cur = self.conn.execute("INSERT INTO saved_route (name) VALUES (?)", (name,))
                self.conn.executemany("INSERT INTO saved_stop VALUES (?, ?, ?)", [(cur.lastrowid, i, s) for i, s in enumerate(stops)])
        except sqlite3.IntegrityError:
            raise DuplicateName(name)
    def get(self, name):
        row = self.conn.execute("SELECT id, name FROM saved_route WHERE name = ?", (name,)).fetchone()
        if row is None:
            return None
        return row[1], [r[0] for r in self.conn.execute("SELECT junction FROM saved_stop WHERE route_id = ? ORDER BY position", (row[0],))]
    def names(self):
        return [r[0] for r in self.conn.execute("SELECT name FROM saved_route ORDER BY name COLLATE NOCASE")]
    def delete(self, name):
        row = self.conn.execute("SELECT id FROM saved_route WHERE name = ?", (name,)).fetchone()
        if row is None:
            return False
        with self.conn:
            self.conn.execute("DELETE FROM saved_stop WHERE route_id = ?", (row[0],))
            self.conn.execute("DELETE FROM saved_route WHERE id = ?", (row[0],))
        return True

def save_route(repo, network, name, stops):
    name = " ".join(name.split())
    if not name or len(stops) < 2:
        raise ValueError("a route needs a name and at least two stops")
    for stop in stops:
        if stop not in network.heights:
            raise UnknownPlace(stop)
    for a, b in zip(stops, stops[1:]):
        if b not in network.paths[a]:
            raise NoRoute(f"no path from {a} to {b}")
    repo.add(name, stops)
    return {"name": name, "stops": list(stops)}

def handle(method, path, body, repo, network):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\nimport sqlite3\n\ndef test_api():\n    \"every outcome its status\"\n    repo, net = RouteRepository(sqlite3.connect(\":memory:\")), sample_network()\n    body = json.dumps({\"name\": \"Ridge walk\", \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]})\n    assert handle(\"POST\", \"/saved-routes\", body, repo, net) == (201, {\"name\": \"Ridge walk\", \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]})\n    assert handle(\"POST\", \"/saved-routes\", body.replace(\"Ridge walk\", \"ridge WALK\"), repo, net) == (409, {\"error\": \"name taken\"})\n    assert handle(\"POST\", \"/saved-routes\", json.dumps({\"name\": \"x\", \"stops\": [\"Mill\", \"Ridge\"]}), repo, net) == (404, {\"error\": \"no path from Mill to Ridge\"})\n    assert handle(\"POST\", \"/saved-routes\", json.dumps({\"name\": \"x\", \"stops\": [\"Moon\", \"Mill\"]}), repo, net) == (404, {\"error\": \"Moon\"})\n    for bad in (\"{\", json.dumps({\"name\": \"x\"}), json.dumps({\"name\": 3, \"stops\": [\"Mill\", \"Weir\"]}), json.dumps({\"name\": \"x\", \"stops\": [\"Mill\"]})):\n        assert handle(\"POST\", \"/saved-routes\", bad, repo, net)[0] == 400, bad\n    assert handle(\"GET\", \"/saved-routes\", \"\", repo, net) == (200, {\"names\": [\"Ridge walk\"]})\n    assert handle(\"GET\", \"/saved-routes/ridge%20walk\", \"\", repo, net) == (200, {\"name\": \"Ridge walk\", \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]})\n    assert handle(\"DELETE\", \"/saved-routes/Ridge%20walk\", \"\", repo, net) == (204, None)\n    assert handle(\"DELETE\", \"/saved-routes/Ridge%20walk\", \"\", repo, net) == (404, {\"error\": \"not found\"})\n    assert handle(\"GET\", \"/elsewhere\", \"\", repo, net) == (404, {\"error\": \"not found\"})")

### Bug hunt: Two routes with one name

The `UNIQUE` rule compares names exactly, so "Ridge walk" and "ridge walk" are both stored: it prints `saved: 2`. Make the column compare without capitals, so the second insert is refused and it prints `saved: 1`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE saved_route (name TEXT NOT NULL UNIQUE)")
for name in ["Ridge walk", "ridge walk"]:
    try:
        conn.execute("INSERT INTO saved_route VALUES (?)", (name,))
    except sqlite3.IntegrityError:
        pass
print("saved:", conn.execute("SELECT COUNT(*) FROM saved_route").fetchone()[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"COLLATE NOCASE\"\n    expect_output(\"saved: 1\")")

### Bug hunt: A migration run twice

Every start-up runs every migration, so the second start crashes with `table junction already exists`. Record the version and skip what's applied, so the program prints `started` twice.

In [ ]:
import sqlite3

MIGRATIONS = ["CREATE TABLE junction (name TEXT PRIMARY KEY)"]

def start(conn):
    for sql in MIGRATIONS:
        conn.execute(sql)
    print("started")

conn = sqlite3.connect(":memory:")
start(conn)
start(conn)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"apply each once\"\n    expect_output(\"started\\nstarted\")")

### Puzzle: One migration, all or nothing

Arrange the code that applies one migration's statements inside an explicit transaction and records its version, rolling back on any error. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status for saving a route under a name that's already taken
q = ___   # True or False: it's fine to edit migration 2 once it has run on someone's database
r = ___   # the SQL collation that makes "Ridge" and "ridge" compare equal
s = ___   # the position column's job in saved_stop: "order", "speed" or "uniqueness of junctions"
t = ___   # what executescript does to an open transaction: "commits it" or "keeps it open"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 409 and q is False and r.upper() == \"NOCASE\", \"p, q and r\"\n    assert s == \"order\" and t == \"commits it\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: safe migrations

Write `migrate_safely(conn, migrations)`. Keep a table `schema_version (version INTEGER PRIMARY KEY, checksum TEXT NOT NULL)`. First, for every migration already applied, compare the SHA-256 hex digest of its text with the stored checksum and raise `MigrationError("migration N was changed after it was applied")` (given) for the first that differs, or `MigrationError("database is at version N but only M migrations exist")` if the database is ahead of the code. Then apply each pending migration in order: its statements (split on `;`, skipping blank ones) and its version row inside one explicit transaction (`BEGIN` ... `COMMIT`); on any `sqlite3.Error`, roll back and raise `MigrationError("migration N failed: MESSAGE")`, leaving earlier migrations applied. Return the numbers applied.

In [ ]:
import hashlib
import sqlite3

class MigrationError(Exception):
    pass

def migrate_safely(conn, migrations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\nM = [\"CREATE TABLE junction (name TEXT PRIMARY KEY, height_m INTEGER NOT NULL)\",\n     \"CREATE TABLE saved_route (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE COLLATE NOCASE); CREATE TABLE saved_stop (route_id INTEGER, position INTEGER, junction TEXT)\",\n     \"CREATE TABLE walk (id INTEGER PRIMARY KEY, day TEXT NOT NULL); INSERT INTO walk (day) VALUES ('2026-11-01'); CREATE TABLE junction (oops TEXT)\"]\n\ndef _tables(conn):\n    return [r[0] for r in conn.execute(\"SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name\")]\n\ndef _error(conn, migrations):\n    try:\n        migrate_safely(conn, migrations)\n    except MigrationError as e:\n        return str(e)\n    return None\n\ndef test_apply_and_fail():\n    \"all or nothing, per migration\"\n    conn = sqlite3.connect(\":memory:\")\n    assert migrate_safely(conn, M[:2]) == [1, 2] and migrate_safely(conn, M[:2]) == []\n    message = _error(conn, M)\n    assert message == \"migration 3 failed: table junction already exists\", message\n    assert _tables(conn) == [\"junction\", \"saved_route\", \"saved_stop\", \"schema_version\"], \"migration 3 left nothing behind\"\n    fixed = M[:2] + [M[2].replace(\"; CREATE TABLE junction (oops TEXT)\", \"\")]\n    assert migrate_safely(conn, fixed) == [3]\n    assert conn.execute(\"SELECT COUNT(*) FROM walk\").fetchone() == (1,)\n\ndef test_checks():\n    \"edited or missing migrations\"\n    conn = sqlite3.connect(\":memory:\")\n    migrate_safely(conn, M[:2])\n    assert _error(conn, [M[0] + \" \", M[1]]) == \"migration 1 was changed after it was applied\"\n    assert _error(conn, M[:1]) == \"database is at version 2 but only 1 migrations exist\"\n    assert \"BEGIN\" in source")

## Warm-up

`COLLATE NOCASE` makes the database compare names without regard to capitals, both for the `UNIQUE` rule and for lookups. Without it, a walker could save "Morning loop" and "morning loop" as two routes and never find the second one again.

## The sprint and today's story

A build sprint is a week of turning planned stories into working, tested, merged code. Each build day follows the same rhythm, which is worth copying for your own capstone:

1. **Re-read the story and its scenarios.** Write any missing ones first (S3 had one; it needs an unhappy path for a duplicate name and one for a route that isn't walkable).
2. **Change the schema**, with a migration.
3. **Inside out**: domain rules, storage, service, interface, each with its tests.
4. **Merge** a small pull request with green CI, and update the burndown.

S3's scenarios, completed:

```
Scenario: S3 saving a route
  Given the stops Mill, Weir, Farm, Ridge
  When I save them as "Ridge walk"
  Then "Ridge walk" appears in my saved routes with 6 km and 1 h 39 min

Scenario: S3 duplicate name
  Given a saved route called "Ridge walk"
  When I save another route as "ridge WALK"
  Then I am told the name is taken

Scenario: S3 a route that isn't walkable
  Given no path joins Mill and Ridge directly
  When I save the stops Mill, Ridge
  Then I am told there is no path from Mill to Ridge
```

## Migrations

Last week's skeleton created its tables when it started. That stops working the day there's real data: you can't drop and recreate a database with someone's walks in it, and every copy of the database (yours, your tests', a user's) may be at a different stage. A **migration** is one numbered change to the schema (`1: create junction and path`, `2: create saved_route and saved_stop`), kept in the code forever. The database records which migrations it has had in a `schema_version` table, and on start-up the program applies the missing ones **in order**.

Rules that keep migrations safe:

- **Never edit a migration that has been applied** anywhere: write a new one. (The challenge detects edits by storing a hash of each.)
- **Each migration is all or nothing**: run it inside a transaction, so a failure halfway leaves the database as it was. SQLite can roll back schema changes too. Note that `executescript` commits as it goes, so run the statements one by one inside `BEGIN` and `COMMIT` instead.
- **Test migrations** on a copy of real data before they meet real data.

The second call applies nothing: the database knows it's up to date.

## The repository

A saved route is a name and an **ordered** list of stops. Two tables model it: `saved_route (id, name UNIQUE COLLATE NOCASE)` and `saved_stop (route_id, position, junction)`, with the primary key `(route_id, position)` so the order is stored, not assumed. The repository (Week 36) hides the SQL behind methods in the domain's terms: `add(name, stops)`, `get(name)`, `names()`, `delete(name)`. It turns the database's `IntegrityError` for a duplicate name into the domain's own `DuplicateName`, so nothing above it needs to know which database it uses.

## Service and API

Two kinds of checking, in two places:

- **At the edge** (the API handler or the service's first lines): is the input well formed? A name of 1 to 40 characters, letters, digits, spaces, hyphens and apostrophes, with extra spaces tidied; stops as a list of at least two strings.
- **In the domain and service**: does it make sense? Every stop must exist, and each consecutive pair must be joined by a path. The service asks the domain (`network.paths`), then saves inside one transaction, and returns the route's distance, climb and time so the interface can show them.

The API follows Day 3's table: `POST /saved-routes` (201, or 400 for bad input, 404 for an unknown place or missing path, 409 for a taken name), `GET /saved-routes`, `GET /saved-routes/NAME` and `DELETE /saved-routes/NAME` (204 or 404), with the name URL-decoded (`urllib.parse.unquote`).

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Move to migrations**: replace any "create tables on start-up" code with a numbered migration list and `migrate_safely`, and add a test that applies every migration to an empty database.
2. **Build today's story** inside out, with its scenarios (add the unhappy paths first) as tests.
3. **Merge** a small pull request with green CI, and update the sprint burndown.

## Before you go

Write today's actual hours next to the story's estimate in your sprint notes. Tomorrow is S5: logging walks, with dates, summaries and a personal pace.